# FastAPI for Backend AI Applications — Engineering Best Practices

A hands-on, runnable guide to building **production-grade backend APIs for AI/ML applications** with FastAPI.

This notebook is opinionated. It shows the patterns that hold up in real systems and explains *why* they matter, not just *how* to type them.

### What you'll learn
1. **Pydantic v2** — request/response validation, the data contract of your API
2. **Dataclasses vs Pydantic** — picking the right tool for internal vs boundary data
3. **Configuration** — typed settings from environment variables (12-factor)
4. **Logging** — structured, configurable, never `print()`
5. **The Singleton pattern** — DB connections and ML models loaded *once*
6. **Database connections** — async SQLAlchemy 2.0 with proper pooling & sessions
7. **Loading an ML model once** — the FastAPI `lifespan` pattern
8. **Dependency Injection** — FastAPI's killer feature for testability
9. **Concurrency** — async vs threads vs processes, and where inference goes
10. **Project structure** — how to lay out a real service
11. **Testing** — `TestClient`, fixtures, dependency overrides
12. **Cross-cutting concerns** — error handling, middleware, observability

> **How to use this notebook:** many cells run directly here (Pydantic, dataclasses, singletons, logging, threading, even an in-process FastAPI test). Full-server examples are marked **"save to a file -> run with uvicorn"** because a notebook is not a deployment target.

## 0. Setup

Install the stack. In a real project these go in `pyproject.toml` / `requirements.txt` with **pinned versions** — never `pip install` ad hoc into production.

In [ ]:
# Run once. The %pip magic installs into the kernel's own environment.
%pip install -q \
    "fastapi>=0.110" \
    "uvicorn[standard]>=0.29" \
    "pydantic>=2.6" \
    "pydantic-settings>=2.2" \
    "sqlalchemy[asyncio]>=2.0" \
    "aiosqlite>=0.20" \
    "httpx>=0.27"
print("Dependencies installed.")

In [ ]:
import sys, pydantic, fastapi, sqlalchemy
print("Python    :", sys.version.split()[0])
print("Pydantic  :", pydantic.VERSION)
print("FastAPI   :", fastapi.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)

## 1. Pydantic v2 — the data contract at your API boundary

Pydantic models are the **single most important best practice** in FastAPI. Every request body and response is validated against a model. This gives you, for free:

- **Validation** — bad input is rejected with a clear 422 error before your code ever runs
- **Parsing / coercion** — `"123"` -> `123`, ISO strings -> `datetime`
- **Serialization** — Python objects -> JSON, with control over what's exposed
- **Self-documenting API** — OpenAPI / Swagger docs generated from the models
- **Editor autocomplete & static type checking**

The golden rule: **validate at the boundary, trust inside.** Once data passes a Pydantic model, the rest of your code can assume it is clean.

In [ ]:
from enum import Enum
from typing import Annotated
from pydantic import BaseModel, Field, field_validator, ConfigDict


class ModelName(str, Enum):
    # Use Enums for fixed sets of choices -- invalid values are rejected automatically.
    gpt = "gpt"
    llama = "llama"
    mistral = "mistral"


class InferenceRequest(BaseModel):
    # ConfigDict replaces the old inner `class Config` from Pydantic v1.
    model_config = ConfigDict(
        extra="forbid",            # reject unexpected fields instead of silently ignoring
        str_strip_whitespace=True,
    )

    prompt: str = Field(..., min_length=1, max_length=4000,
                        description="The user prompt to send to the model.")
    model: ModelName = ModelName.gpt
    # Annotated + Field is the modern way to attach constraints to a type.
    temperature: Annotated[float, Field(ge=0.0, le=2.0)] = 0.7
    max_tokens: Annotated[int, Field(gt=0, le=8192)] = 512

    @field_validator("prompt")
    @classmethod
    def prompt_not_blank(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("prompt cannot be blank")
        return v


# A valid request -- note temperature given as a string gets coerced to float.
req = InferenceRequest(prompt="Explain singletons.", temperature="0.4", max_tokens=256)
print("Parsed OK:", req)
print("Type of temperature:", type(req.temperature).__name__)

In [ ]:
from pydantic import ValidationError

# Watch Pydantic reject bad input with precise, structured errors.
bad_inputs = [
    {"prompt": "", "temperature": 0.5},        # blank prompt
    {"prompt": "hi", "temperature": 5.0},      # temperature out of range
    {"prompt": "hi", "model": "gpt-9000"},     # not a valid enum member
    {"prompt": "hi", "unexpected_field": 123}, # extra field forbidden
]

for data in bad_inputs:
    try:
        InferenceRequest(**data)
    except ValidationError as e:
        first = e.errors()[0]
        print(f"REJECTED {str(data):55.55}  ->  {first['loc']}: {first['msg']}")

### Response models, and *never leak internal fields*

A common security/clarity bug: returning your full DB object (with password hashes, internal flags, etc.). Define an explicit **response model** so only intended fields go out. FastAPI uses `response_model` to filter the output.

In [ ]:
from pydantic import EmailStr

class UserInDB(BaseModel):
    # What we store internally -- includes secrets.
    id: int
    email: str
    hashed_password: str           # MUST NOT be exposed
    is_admin: bool = False

class UserPublic(BaseModel):
    # What the API returns -- a safe subset.
    id: int
    email: str

internal = UserInDB(id=1, email="a@b.com", hashed_password="$argon2id$...", is_admin=True)

# Build the public view from the internal object. model_dump() -> plain dict.
public = UserPublic(**internal.model_dump())
print("Internal :", internal.model_dump())
print("Exposed  :", public.model_dump())     # no password, no is_admin
print("JSON out :", public.model_dump_json())

**Key Pydantic v2 method/name changes to remember:**

| Old (v1) | New (v2) |
|---|---|
| `.dict()` | `.model_dump()` |
| `.json()` | `.model_dump_json()` |
| `parse_obj()` | `model_validate()` |
| `class Config:` | `model_config = ConfigDict(...)` |
| `@validator` | `@field_validator` |
| `@root_validator` | `@model_validator` |

Use `.model_dump(exclude_none=True)`, `exclude={"field"}`, or `by_alias=True` to control serialization precisely.

## 2. Dataclasses vs Pydantic — use the right tool

Both bundle data, but they solve different problems:

- **Pydantic** — for data crossing a *trust boundary* (API requests/responses, config, external data). It **validates and parses**. Costs a little overhead.
- **`@dataclass`** — for *internal*, already-trusted structures (passing things between your own functions, return types, value objects). Zero validation, near-zero overhead, standard library.

Rule of thumb: **Pydantic at the edges, dataclasses in the core.** Don't re-validate the same data ten times as it moves through your own code.

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True, slots=True)   # frozen=immutable & hashable; slots=less memory, faster attr access
class InferenceResult:
    # An internal value object. Already trusted, so no validation needed.
    text: str
    tokens_used: int
    model: str
    latency_ms: float
    metadata: dict = field(default_factory=dict)   # never use a mutable default directly!

r = InferenceResult(text="42", tokens_used=3, model="gpt", latency_ms=120.5)
print(r)

# frozen=True prevents mutation, killing a whole class of bugs:
try:
    r.text = "changed"
except Exception as e:
    print("Immutable:", type(e).__name__, "-", e)

> **Gotcha shown above:** never write `metadata: dict = {}` as a default — that mutable default is *shared across all instances*. Use `field(default_factory=dict)`. The same trap applies to plain function args (`def f(x=[])` is the classic Python footgun).

If you want dataclass *syntax* but Pydantic *validation*, use `pydantic.dataclasses.dataclass` — a lightweight validated struct without the full `BaseModel` API.

## 3. Configuration — typed settings from the environment

**Never hard-code secrets, hosts, or model paths.** The [12-factor](https://12factor.net/config) principle: configuration lives in the *environment*, code is identical across dev/staging/prod.

`pydantic-settings` gives you a typed, validated `Settings` object loaded from environment variables (and/or a `.env` file). Benefits:

- Missing required config fails **loudly at startup**, not mysteriously at request time
- Types are validated (a bad port number is caught immediately)
- One obvious place to see everything the app needs

We load `Settings` **once** and reuse it (a singleton — see Section 5).

In [ ]:
import os
from functools import lru_cache
from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",          # optional: read from a .env file
        env_prefix="APP_",        # env vars are APP_DATABASE_URL, APP_LOG_LEVEL, ...
        case_sensitive=False,
        extra="ignore",
    )

    app_name: str = "ai-backend"
    environment: str = "development"
    debug: bool = False

    database_url: str = "sqlite+aiosqlite:///./app.db"
    log_level: str = "INFO"

    model_path: str = "/models/my-model.bin"
    max_batch_size: int = Field(default=16, ge=1, le=512)

    # Secrets: provide via env, never a default in code.
    api_secret_key: str = "dev-only-change-me"


@lru_cache  # <-- cached: Settings() is constructed once, reused everywhere
def get_settings() -> Settings:
    return Settings()


# Simulate environment variables being set (in prod these come from the OS/container).
os.environ["APP_ENVIRONMENT"] = "production"
os.environ["APP_MAX_BATCH_SIZE"] = "32"

get_settings.cache_clear()           # clear so the new env vars take effect in this demo
settings = get_settings()
print("Loaded settings:")
for k, v in settings.model_dump().items():
    shown = "***" if "secret" in k or "key" in k else v
    print(f"  {k:18} = {shown}")

print("\nSame object every call?", get_settings() is get_settings())

`@lru_cache` on `get_settings` is the simplest, most Pythonic singleton: the function runs once, the result is cached, every later call returns the same object. In FastAPI you inject it with `Depends(get_settings)` (Section 8), which also makes it trivially overridable in tests.

## 4. Logging — structured, configured, and never `print()`

`print()` is invisible to log aggregators, has no levels, no timestamps, no context, and can't be turned off. **Use the `logging` module.**

Best practices:
- One logger per module: `logger = logging.getLogger(__name__)`. The name tells you *where* a line came from.
- Configure logging **once**, centrally (via `dictConfig`), at startup.
- Log **levels** mean something: DEBUG (dev detail), INFO (normal events), WARNING (something odd but handled), ERROR (a request failed), CRITICAL (the app is in trouble).
- In production prefer **structured / JSON logs** so fields are queryable (e.g. by `request_id`, `user_id`, `latency_ms`).
- Never log secrets or full PII. Never log full request bodies blindly.
- Use `logger.exception(...)` inside an `except` block to capture the traceback.

In [ ]:
import logging
import logging.config

# Central configuration -- call this ONCE at application startup.
LOGGING_CONFIG = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "standard": {
            "format": "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            "datefmt": "%Y-%m-%d %H:%M:%S",
        },
    },
    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "formatter": "standard",
            "level": "DEBUG",
            "stream": "ext://sys.stdout",
        },
    },
    "root": {"handlers": ["console"], "level": "INFO"},
    "loggers": {
        # Quiet noisy third-party loggers; let your own app log at INFO/DEBUG.
        "uvicorn.access": {"level": "WARNING"},
        "sqlalchemy.engine": {"level": "WARNING"},
    },
}
logging.config.dictConfig(LOGGING_CONFIG)

# Per-module logger -- this is the pattern you repeat in every file.
logger = logging.getLogger("ai_backend.demo")

logger.debug("This won't show -- root level is INFO.")
logger.info("Model warm-up started", extra={"model": "gpt"})
logger.warning("Fallback model used; primary unavailable")

try:
    1 / 0
except ZeroDivisionError:
    logger.exception("Inference failed")   # logs message + full traceback at ERROR level

### Structured logging with request context

For real services you want each line tagged with a **request id** so you can trace one request across many log lines. Below is a minimal JSON formatter plus a `contextvars`-based request id — the same idea libraries like `structlog` give you with more polish.

In [ ]:
import json, logging, uuid
from contextvars import ContextVar

# A request-scoped id, set by middleware (Section 12) and read by the formatter.
request_id_var: ContextVar[str] = ContextVar("request_id", default="-")

class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        payload = {
            "ts": self.formatTime(record, "%Y-%m-%dT%H:%M:%S"),
            "level": record.levelname,
            "logger": record.name,
            "msg": record.getMessage(),
            "request_id": request_id_var.get(),
        }
        if record.exc_info:
            payload["exc"] = self.formatException(record.exc_info)
        return json.dumps(payload)

# Demo: attach the JSON formatter to a fresh logger.
demo = logging.getLogger("ai_backend.json_demo")
demo.handlers.clear()
h = logging.StreamHandler()
h.setFormatter(JsonFormatter())
demo.addHandler(h)
demo.setLevel(logging.INFO)
demo.propagate = False

request_id_var.set(str(uuid.uuid4())[:8])
demo.info("handling inference request")
demo.info("inference complete")   # same request_id ties these together

## 5. The Singleton pattern — one instance, shared everywhere

A **singleton** ensures a class has exactly one instance, reused across the whole app. In backend AI services you use it for **expensive, shared resources** that should be created once:

- the **database engine / connection pool**
- a loaded **ML model** (multi-GB; reloading per request is fatal to performance)
- an HTTP client with connection pooling
- the settings object

Three ways to do this in Python, from most to least Pythonic. Pick based on context.

In [ ]:
# --- Approach A: a module-level instance (the most Pythonic singleton) ---
# A module is imported once and cached in sys.modules, so anything defined at
# module level is effectively a singleton. Often this is all you need.

class _ConnectionPool:
    def __init__(self):
        print("  >> Expensive pool created (happens once)")
        self.connections = 10

# Created at import time; every importer shares THIS object.
pool = _ConnectionPool()
print("module-level singleton:", pool, "with", pool.connections, "connections")

In [ ]:
# --- Approach B: __new__ override (classic OOP singleton) ---
import threading

class DatabaseClient:
    _instance = None
    _lock = threading.Lock()      # thread-safe creation (double-checked locking)

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:          # check again inside the lock
                    cls._instance = super().__new__(cls)
                    cls._instance._initialized = False
        return cls._instance

    def __init__(self, url: str = "sqlite://"):
        if self._initialized:                      # guard so __init__ runs only once
            return
        self.url = url
        self._initialized = True
        print(f"  >> DatabaseClient connected to {url} (init runs once)")

a = DatabaseClient("postgres://primary")
b = DatabaseClient("postgres://ignored")     # __init__ body is skipped
print("Same instance?", a is b, "| url:", b.url)

In [ ]:
# --- Approach C: a metaclass (reusable across many classes) ---
class SingletonMeta(type):
    _instances: dict = {}
    _lock = threading.Lock()

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            with cls._lock:
                if cls not in cls._instances:
                    cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class ModelRegistry(metaclass=SingletonMeta):
    def __init__(self):
        print("  >> ModelRegistry created (once)")
        self.models = {}

x = ModelRegistry()
y = ModelRegistry()
print("Same registry?", x is y)

**Which to use?**

| Approach | When |
|---|---|
| **Module-level instance** | Default. Simple, obvious, thread-safe by import semantics. |
| **`@lru_cache` on a factory** | When you want lazy creation + easy override in tests (see Settings, Section 3). **Best for FastAPI dependencies.** |
| **`__new__` override** | When callers will naturally write `DatabaseClient()` and must get the same object. |
| **Metaclass** | When you need the *same* singleton behaviour across many classes. Often overkill. |

> **Important caveat for FastAPI:** with multiple worker processes (`uvicorn --workers 4`), each *process* gets its own singleton — they are not shared across processes. That's fine for read-only resources like a loaded model (each worker loads its own copy). For shared mutable state across workers, use an external store (Redis, the database). For per-request resources, prefer **dependency injection over global singletons** because DI is testable; reserve singletons for genuinely process-global, expensive objects.

## 6. Database connections — async SQLAlchemy 2.0 done right

Best practices for talking to a database from an async API:

1. **One engine per process** (it owns the connection pool) — a singleton.
2. **One session per request** — short-lived, opened at the start of the request, closed at the end. Never share a session across requests or threads.
3. Use a **connection pool** (SQLAlchemy manages this); tune `pool_size` / `max_overflow` for your load.
4. Use the **async** driver (`asyncpg` for Postgres, `aiosqlite` for SQLite) so DB I/O doesn't block the event loop.
5. Inject the session as a **dependency** so endpoints don't manage lifecycle and tests can swap it out.

This cell creates the engine and an in-memory schema so we can actually run queries.

In [ ]:
import asyncio
from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker, AsyncSession
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from sqlalchemy import String, select

# --- Engine: created ONCE (singleton). Owns the connection pool. ---
engine = create_async_engine(
    "sqlite+aiosqlite:///:memory:",
    echo=False,
    pool_pre_ping=True,     # checks a connection is alive before using it (avoids stale-connection errors)
    # For Postgres you'd also set: pool_size=10, max_overflow=20
)

# --- Session factory: a callable that produces fresh sessions. ---
SessionFactory = async_sessionmaker(engine, expire_on_commit=False, class_=AsyncSession)

# --- ORM models (SQLAlchemy 2.0 typed style) ---
class Base(DeclarativeBase):
    pass

class Conversation(Base):
    __tablename__ = "conversations"
    id:     Mapped[int] = mapped_column(primary_key=True)
    user:   Mapped[str] = mapped_column(String(64), index=True)
    prompt: Mapped[str] = mapped_column(String(4000))

async def init_db():
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)

await init_db()   # top-level await works in Jupyter
print("Database ready, schema created.")

In [ ]:
# The per-request session dependency. `async with` guarantees the session is
# always closed, even on error. This is the function FastAPI will call via Depends.
from collections.abc import AsyncGenerator

async def get_session() -> AsyncGenerator[AsyncSession, None]:
    async with SessionFactory() as session:
        try:
            yield session
            await session.commit()        # commit if the request succeeded
        except Exception:
            await session.rollback()      # roll back on any failure
            raise
        # `async with` closes/returns the connection to the pool here.

# Demonstrate it directly (FastAPI would drive this for you per request).
async def demo_db():
    agen = get_session()
    session = await agen.__anext__()
    try:
        session.add_all([
            Conversation(user="alice", prompt="What is a singleton?"),
            Conversation(user="bob",   prompt="Explain async vs threads."),
        ])
        await session.flush()
        rows = (await session.execute(select(Conversation))).scalars().all()
        print("Rows in DB:")
        for r in rows:
            print(f"  #{r.id} {r.user}: {r.prompt}")
    finally:
        await agen.aclose()   # triggers commit + cleanup

await demo_db()

Notice the **separation of concerns**: the engine/pool is a process-wide singleton, while each request gets its own session from the factory. The `get_session` generator is exactly the shape FastAPI's dependency system expects — code before `yield` is setup, code after is teardown.

## 7. Loading an ML model **once** — the `lifespan` pattern

The #1 performance mistake in AI backends: **loading the model inside the request handler.** A model can take seconds to load and gigabytes of RAM. Loading it per request makes every request catastrophically slow.

**Load it once at startup, keep it in memory, reuse it for every request.**

FastAPI's modern mechanism is the **`lifespan` context manager** (it replaced the deprecated `@app.on_event("startup")`). Code before `yield` runs once on startup; code after runs once on shutdown. You stash the loaded model where handlers can reach it — on `app.state` or in a module-level container.

In [ ]:
import time

# --- A stand-in for a real heavy model (transformers, torch, etc.) ---
class SentimentModel:
    def __init__(self, path: str):
        print(f"  >> Loading model weights from {path} ...")
        time.sleep(1.0)            # simulate a slow load
        self.path = path
        self.loaded_at = time.time()
        print("  >> Model loaded.")

    def predict(self, text: str) -> dict:
        # Pretend inference. Real models do heavy CPU/GPU work here.
        score = (len(text) % 10) / 10
        return {"label": "positive" if score > 0.5 else "negative", "score": round(score, 3)}


# A container that lives for the process lifetime. Cleared on shutdown.
ml_models: dict[str, object] = {}

print("Simulating startup (loads ONCE):")
ml_models["sentiment"] = SentimentModel("/models/sentiment.bin")

print("\nNow every request reuses the SAME object -- no reload:")
for text in ["I love this", "meh"]:
    print(f"  predict({text!r}) ->", ml_models["sentiment"].predict(text))

print("\nSame object across calls?",
      ml_models["sentiment"] is ml_models["sentiment"])

Here is exactly how that wires into FastAPI. **Save this as `app/main.py` and run `uvicorn app.main:app`** — it won't fully run inside a notebook because it's a server, but the structure is the whole point.

In [ ]:
app_main_code = r"""
# app/main.py  --  save to a file, run with:  uvicorn app.main:app --reload
import logging
from contextlib import asynccontextmanager
from fastapi import FastAPI, Depends

logger = logging.getLogger(__name__)
ml_models: dict[str, object] = {}     # process-global model container


@asynccontextmanager
async def lifespan(app: FastAPI):
    # ---- startup: runs ONCE before the server accepts requests ----
    logger.info("Loading ML model...")
    ml_models["sentiment"] = SentimentModel("/models/sentiment.bin")
    # you'd also create the DB engine here, warm caches, etc.
    yield
    # ---- shutdown: runs ONCE as the server stops ----
    logger.info("Releasing resources...")
    ml_models.clear()
    # await engine.dispose()


app = FastAPI(title="AI Backend", lifespan=lifespan)


# Inject the model via a dependency instead of reaching for a global directly --
# this keeps handlers testable (you can override get_model in tests).
def get_model() -> "SentimentModel":
    return ml_models["sentiment"]


@app.post("/predict")
async def predict(req: InferenceRequest, model: SentimentModel = Depends(get_model)):
    return model.predict(req.prompt)


@app.get("/healthz")
async def healthz():
    # readiness check: confirm the model is actually loaded
    return {"status": "ok", "model_loaded": "sentiment" in ml_models}
"""
print(app_main_code)

> **Why a dict / `app.state` and not just a module global?** Either works, but routing access through a **dependency** (`Depends(get_model)`) means tests can override `get_model` to return a fake — no monkey-patching globals. That's the recurring theme: singletons for the *resource*, dependency injection for *access*.

## 8. Dependency Injection — FastAPI's superpower

DI means a function declares *what it needs* and FastAPI *provides it*. You've already seen `get_session`, `get_settings`, `get_model`. Benefits:

- **Testability** — override any dependency with a fake in tests (`app.dependency_overrides`).
- **Reuse** — auth, DB sessions, pagination params written once, used everywhere.
- **Composition** — dependencies can depend on other dependencies.
- **Lifecycle** — generator dependencies (`yield`) get automatic setup/teardown.

Common DI patterns: shared query params, authentication, and "current user".

In [ ]:
from typing import Annotated
from fastapi import Depends, HTTPException, Header

# 1) Reusable pagination params -- inject into many endpoints.
class Pagination:
    def __init__(self, limit: int = 20, offset: int = 0):
        self.limit = min(limit, 100)     # cap to protect the DB
        self.offset = max(offset, 0)

# 2) Auth dependency -- raises 401 if the token is bad; returns the user otherwise.
async def get_current_user(authorization: Annotated[str | None, Header()] = None) -> dict:
    if authorization != "Bearer secret-token":
        raise HTTPException(status_code=401, detail="Invalid or missing token")
    return {"id": 1, "email": "alice@example.com"}

# 3) A dependency that DEPENDS on another dependency (composition).
async def require_admin(user: Annotated[dict, Depends(get_current_user)]) -> dict:
    if user["id"] != 1:
        raise HTTPException(status_code=403, detail="Admin only")
    return user

# Annotated aliases keep signatures clean and reusable:
CurrentUser = Annotated[dict, Depends(get_current_user)]
Page        = Annotated[Pagination, Depends(Pagination)]

print("Dependencies defined. In an endpoint you'd write:")
print("    async def list_items(user: CurrentUser, page: Page): ...")
print("FastAPI resolves `user` (running auth) and `page` automatically per request.")

## 9. Concurrency — async vs threads vs processes (and where inference goes)

This is the part most people get wrong in AI backends. Three tools, three jobs:

| Tool | Good for | Why |
|---|---|---|
| **`async`/`await`** | **I/O-bound** waiting: DB queries, HTTP calls, reading files | One thread juggles thousands of waits without blocking. This is FastAPI's native model. |
| **Threads** (`ThreadPoolExecutor`, `run_in_threadpool`) | **Blocking** calls you can't make async (a sync DB driver, a library that does blocking I/O, light CPU work) | Keeps the event loop free while the blocking call runs elsewhere. The GIL limits true CPU parallelism, but releases during I/O and many native ops (NumPy, torch). |
| **Processes** (`ProcessPoolExecutor`, or an external worker queue) | **CPU-bound** work: heavy pure-Python compute | Separate processes sidestep the GIL for real parallelism. |

**The cardinal rule:** *never run blocking or CPU-heavy code directly in an `async def` endpoint* — it freezes the event loop and stalls every other request.

**Where does ML inference go?** Inference is usually CPU/GPU-heavy and *blocking*. Options, in order of typical preference:
1. Offload to a **thread pool** (`run_in_threadpool`) — simplest; works well because torch/NumPy release the GIL during the heavy native math, so other requests keep flowing.
2. Offload to a **process pool** for pure-Python CPU work that holds the GIL.
3. For real scale, push inference to a **dedicated worker / queue** (Celery, Ray, Triton, a separate inference service) and keep the API thin.

In [ ]:
import asyncio, time, concurrent.futures

# A blocking, CPU-ish function -- imagine this is model inference.
def blocking_inference(x: int) -> int:
    total = 0
    for _ in range(2_000_000):     # busy work to simulate compute
        total += x
    return total

# WRONG: calling this directly in an async endpoint blocks the whole event loop.
# RIGHT: offload it so the loop stays responsive.

async def good_endpoint(x: int) -> int:
    loop = asyncio.get_running_loop()
    # run_in_executor offloads the blocking call to a thread pool.
    # (FastAPI does this automatically for `def` endpoints; for `async def` you do it yourself.)
    return await loop.run_in_executor(None, blocking_inference, x)

async def main():
    t0 = time.perf_counter()
    # These run concurrently because each is offloaded off the event loop.
    results = await asyncio.gather(*[good_endpoint(i) for i in range(4)])
    dt = time.perf_counter() - t0
    print("results:", results)
    print(f"4 offloaded tasks finished in {dt:.2f}s (overlapped, loop never blocked)")

await main()

In FastAPI specifically, there's an elegant shortcut: **the type of your path function decides the execution model.**

- `async def endpoint(...)` -> runs **on the event loop**. Only put `await`-able I/O here. *Never* call blocking code.
- `def endpoint(...)` (plain, no `async`) -> FastAPI automatically runs it **in a thread pool**. Perfect for a blocking model call or a sync library.

So the easiest correct pattern for a blocking inference handler is often just to make it a plain `def`.

In [ ]:
fastapi_concurrency_code = r"""
# Concurrency patterns inside FastAPI -- save & run with uvicorn.
import asyncio
from fastapi import FastAPI
from fastapi.concurrency import run_in_threadpool

app = FastAPI()

# (1) Pure async I/O: stays on the event loop. Correct use of async.
@app.get("/io")
async def io_bound():
    await asyncio.sleep(0.1)            # e.g. an awaited DB / HTTP call
    return {"ok": True}

# (2) Blocking call inside an async handler: offload explicitly.
@app.post("/infer-async")
async def infer_async(x: int):
    result = await run_in_threadpool(blocking_inference, x)   # <- key line
    return {"result": result}

# (3) Simplest correct pattern for blocking work: just use `def`.
#     FastAPI auto-runs plain `def` handlers in its thread pool.
@app.post("/infer-sync")
def infer_sync(x: int):
    return {"result": blocking_inference(x)}

# (4) Truly CPU-bound pure-Python work that holds the GIL -> a process pool.
from concurrent.futures import ProcessPoolExecutor
pool = ProcessPoolExecutor(max_workers=4)   # create once, reuse (a singleton!)

@app.post("/infer-cpu")
async def infer_cpu(x: int):
    loop = asyncio.get_running_loop()
    result = await loop.run_in_executor(pool, blocking_inference, x)
    return {"result": result}
"""
print(fastapi_concurrency_code)

### Background tasks and fire-and-forget

For work that should happen *after* responding (sending an email, writing an audit log, kicking off a slow job), use `BackgroundTasks`. For heavy or retryable jobs, use a real queue (Celery / RQ / Arq) instead — `BackgroundTasks` runs in the same process and dies if the worker restarts.

In [ ]:
background_tasks_code = r"""
from fastapi import BackgroundTasks, FastAPI
app = FastAPI()

def write_audit_log(user_id: int, action: str):
    # runs AFTER the response is sent; keep it quick & non-critical
    ...

@app.post("/predict")
async def predict(user_id: int, tasks: BackgroundTasks):
    result = {"label": "positive"}
    tasks.add_task(write_audit_log, user_id, "predict")   # fire-and-forget
    return result   # client gets this immediately; audit log runs after
"""
print(background_tasks_code)

## 10. Project structure — how a real service is laid out

A flat `main.py` is fine for a demo. Real services separate concerns so the codebase stays navigable and testable. A common, scalable layout:

```
my-ai-service/
├── pyproject.toml            # deps + tool config (ruff, mypy, pytest), pinned versions
├── .env.example              # documents required env vars (never commit real .env)
├── Dockerfile
├── app/
│   ├── __init__.py
│   ├── main.py               # FastAPI app, lifespan, router registration
│   ├── config.py             # Settings (pydantic-settings) + get_settings()
│   ├── logging_config.py     # dictConfig setup
│   ├── api/                  # the HTTP layer (routers only -- thin!)
│   │   ├── deps.py           #   shared dependencies (get_session, get_current_user)
│   │   └── routes/
│   │       ├── health.py
│   │       └── inference.py
│   ├── schemas/              # Pydantic request/response models (the API contract)
│   │   └── inference.py
│   ├── models/               # SQLAlchemy ORM models (DB tables)
│   │   └── conversation.py
│   ├── services/             # business logic (no FastAPI imports here!)
│   │   └── inference_service.py
│   ├── ml/                   # model loading & inference wrappers
│   │   └── sentiment.py
│   └── db/
│       ├── engine.py         # engine + SessionFactory (singletons)
│       └── base.py           # DeclarativeBase
└── tests/
    ├── conftest.py           # fixtures, dependency overrides
    └── test_inference.py
```

**The layering rule (most important takeaway):**

```
api/routes  ->  services  ->  db / ml
(HTTP only)     (logic)       (resources)
```

- **Routers** parse/validate input (Pydantic), call a service, shape the response. They contain *no business logic*.
- **Services** hold the actual logic and know nothing about HTTP. This makes them unit-testable without a web server and reusable from CLIs, workers, etc.
- **db / ml** layers own the resources (engine, models).

This separation is what lets you test business logic in isolation and swap implementations without rewriting endpoints.

## 11. Testing — `TestClient` + dependency overrides

Because we used dependency injection everywhere, testing is clean: spin up the app in-process with `TestClient` and **override dependencies** to inject fakes (a fake model, an in-memory DB). No network, no real model, fast and deterministic.

This cell builds a tiny app and actually runs tests against it, right here.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

# --- A small but realistic app demonstrating the layering ---
class PredictIn(BaseModel):
    text: str

class PredictOut(BaseModel):
    label: str
    score: float

# resource layer (would be a singleton loaded in lifespan)
class RealModel:
    def predict(self, text: str) -> dict:
        return {"label": "positive", "score": 0.99}

def get_model() -> RealModel:           # the dependency we'll override in tests
    return RealModel()

app = FastAPI()

@app.post("/predict", response_model=PredictOut)
async def predict(body: PredictIn, model: RealModel = Depends(get_model)):
    if not body.text.strip():
        raise HTTPException(422, "text required")
    return model.predict(body.text)

# --- Tests ---
client = TestClient(app)

# 1) Happy path against the real dependency
r = client.post("/predict", json={"text": "hello"})
print("real model     :", r.status_code, r.json())

# 2) Override the model dependency with a deterministic fake
class FakeModel:
    def predict(self, text: str) -> dict:
        return {"label": "negative", "score": 0.123}

app.dependency_overrides[get_model] = lambda: FakeModel()
r = client.post("/predict", json={"text": "hello"})
print("overridden fake:", r.status_code, r.json())
app.dependency_overrides.clear()        # always clean up overrides

# 3) Validation error path (Pydantic rejects missing field -> 422)
r = client.post("/predict", json={})
print("missing field  :", r.status_code, "->", r.json()["detail"][0]["msg"])

assert client.post("/predict", json={"text": "x"}).status_code == 200
print("\nAll assertions passed.")

That `app.dependency_overrides` mechanism is *why* we route everything through `Depends`. In a real `tests/conftest.py` you'd define pytest fixtures that override the DB session to point at an in-memory SQLite, override `get_model` to a fake, and yield a configured `TestClient`. Async endpoints that need a real event loop can be tested with `httpx.AsyncClient` + `pytest-asyncio`.

## 12. Cross-cutting concerns — errors, middleware, observability

The polish that separates a toy from a production service.

**Error handling.** Don't let raw exceptions leak stack traces to clients. Define custom exceptions in your service layer and a global handler that maps them to clean HTTP responses with appropriate status codes. Log the full detail server-side; return a safe message client-side.

**Middleware** runs on every request — ideal for: assigning a request id, logging latency, adding security headers, CORS, and metrics.

**Observability.** Three pillars: structured **logs** (Section 4), **metrics** (Prometheus: request count, latency histograms, model queue depth), and **traces** (OpenTelemetry across services). At minimum expose a `/healthz` (liveness) and `/readyz` (readiness — is the model loaded?) so your orchestrator knows when the pod can serve traffic.

In [ ]:
cross_cutting_code = r"""
# Error handling + middleware -- the production-hardening layer.
import time, uuid, logging
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware

logger = logging.getLogger(__name__)
app = FastAPI()

# --- Domain exceptions live in the service layer (no HTTP knowledge) ---
class ModelOverloadedError(Exception): ...
class ResourceNotFoundError(Exception): ...

# --- Global handlers map domain errors -> clean HTTP responses ---
@app.exception_handler(ModelOverloadedError)
async def overloaded_handler(request: Request, exc: ModelOverloadedError):
    return JSONResponse(status_code=503, content={"detail": "Model busy, retry shortly"})

@app.exception_handler(ResourceNotFoundError)
async def not_found_handler(request: Request, exc: ResourceNotFoundError):
    return JSONResponse(status_code=404, content={"detail": str(exc)})

@app.exception_handler(Exception)                       # catch-all safety net
async def unhandled_handler(request: Request, exc: Exception):
    logger.exception("Unhandled error")                 # full traceback server-side
    return JSONResponse(status_code=500, content={"detail": "Internal server error"})

# --- Middleware: request id + latency logging on every request ---
@app.middleware("http")
async def add_request_context(request: Request, call_next):
    request_id = str(uuid.uuid4())[:8]
    request_id_var.set(request_id)        # ties all log lines for this request together
    start = time.perf_counter()
    response = await call_next(request)
    elapsed_ms = (time.perf_counter() - start) * 1000
    response.headers["X-Request-ID"] = request_id
    logger.info("%s %s -> %s (%.1fms)",
                request.method, request.url.path, response.status_code, elapsed_ms)
    return response

# --- CORS (only if a browser frontend calls you directly) ---
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://your-frontend.example.com"],   # never "*" in prod with credentials
    allow_methods=["*"],
    allow_headers=["*"],
)
"""
print(cross_cutting_code)

## 13. The checklist — best practices in one place

**Data & validation**
- Pydantic models on every request and response; `extra="forbid"` on inputs.
- Explicit `response_model` so internal fields never leak.
- Dataclasses for internal value objects; Pydantic only at trust boundaries.

**Configuration**
- All config from the environment via `pydantic-settings`; secrets never in code.
- Load `Settings` once (`@lru_cache`); inject via `Depends`.

**Logging & observability**
- `logging` module, never `print`; one logger per module (`getLogger(__name__)`).
- Configure once with `dictConfig`; structured/JSON logs with a request id in prod.
- `/healthz` + `/readyz`; metrics (Prometheus) and traces (OpenTelemetry) at scale.

**Resources & singletons**
- DB engine/pool and ML model created **once** per process.
- Load models in **`lifespan`** startup — never inside a request handler.
- Singleton for the *resource*; dependency injection for *access* (testability).

**Database**
- Async driver + connection pool; `pool_pre_ping=True`.
- One session per request via a generator dependency (commit/rollback/close).
- Never share a session across requests or threads.

**Concurrency**
- `async def` only for awaitable I/O; never block the event loop.
- Blocking/inference work -> plain `def` handler or `run_in_threadpool`.
- Pure-Python CPU work -> process pool; heavy/retryable jobs -> external queue.

**Structure & testing**
- Layer it: routes (HTTP) -> services (logic) -> db/ml (resources).
- Services contain no FastAPI imports, so they unit-test in isolation.
- `TestClient` + `dependency_overrides` to inject fakes; in-memory DB for tests.

**Hardening**
- Global exception handlers map domain errors to clean HTTP responses; log detail server-side, return safe messages.
- Middleware for request id, latency logging, security headers, CORS.
- Pin dependencies; run `ruff` + `mypy` + `pytest` in CI; ship in a container with a non-root user.

---

### Where to go next
- Wire these pieces into the project structure from Section 10 and run it with `uvicorn app.main:app --reload`.
- Add auth (OAuth2 / JWT) using the dependency pattern from Section 8.
- Containerize, add a `/metrics` endpoint, and put a real inference engine behind the service layer.

You now have the full set of engineering best practices for a production AI backend. The throughline: **validate at the edges, load expensive things once, inject dependencies for testability, never block the event loop, and keep layers separate.**